# Imports

In [81]:
import numpy as np
import joblib

from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Load Feature Extractor

In [82]:
base_model = InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)

feature_extractor = Model(inputs=base_model.input, outputs=x)

print("_/_/ Feature extractor ready")

_/_/ Feature extractor ready


# Data Generators

In [83]:
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_directory(
    'dataset/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

validation_generator = datagen.flow_from_directory(
    'dataset/validation',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

test_generator = datagen.flow_from_directory(
    'dataset/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

print("Classes:", train_generator.class_indices)

Found 1145 images belonging to 6 classes.
Found 267 images belonging to 6 classes.
Found 220 images belonging to 6 classes.
Classes: {'A1-Walking': 0, 'A2-Sitting-down': 1, 'A3-StandUp': 2, 'A4-PickObject': 3, 'A5-DrinkWater': 4, 'A6-Fall': 5}


# Feature Extraction Function

In [84]:
def extract_features(generator, model):
    features = []
    labels = []

    for i in range(len(generator)):
        x_batch, y_batch = generator[i]
        f_batch = model.predict(x_batch, verbose=0)

        features.append(f_batch)
        labels.append(np.argmax(y_batch, axis=1))

    features = np.vstack(features)
    labels = np.hstack(labels)

    return features, labels

# Extract Features

In [85]:
print("Extracting TRAIN features...")
X_train, y_train = extract_features(train_generator, feature_extractor)

print("Extracting VALIDATION features...")
X_val, y_val = extract_features(validation_generator, feature_extractor)

print("Extracting TEST features...")
X_test, y_test = extract_features(test_generator, feature_extractor)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

Extracting TRAIN features...
Extracting VALIDATION features...
Extracting TEST features...
Shapes: (1145, 2048) (267, 2048) (220, 2048)


# Normalize Features (CRITICAL for KNN)

In [86]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("_/_/ Features normalized")

_/_/ Features normalized


# Train KNN

In [87]:
print("Training KNN...")

knn_model = KNeighborsClassifier(
    n_neighbors=69,
    metric='euclidean'
)

knn_model.fit(X_train, y_train)

print("_/_/ KNN trained")

Training KNN...
_/_/ KNN trained


# Validation Evaluation

In [88]:
y_val_pred = knn_model.predict(X_val)

val_acc = accuracy_score(y_val, y_val_pred)
print("Validation Accuracy:", val_acc)

Validation Accuracy: 0.7752808988764045


# Test Evaluation

In [89]:
y_test_pred = knn_model.predict(X_test)

test_acc = accuracy_score(y_test, y_test_pred)

print("_/_/ Final Test Accuracy:", test_acc)

print("\nClassification Report:\n")
print(classification_report(y_test, y_test_pred))

_/_/ Final Test Accuracy: 0.6954545454545454

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.97      0.97        40
           1       0.81      0.62      0.70        40
           2       0.48      0.60      0.53        40
           3       0.54      0.62      0.58        40
           4       0.67      0.50      0.57        40
           5       0.87      1.00      0.93        20

    accuracy                           0.70       220
   macro avg       0.72      0.72      0.72       220
weighted avg       0.71      0.70      0.70       220



# Save Model

In [90]:
joblib.dump(knn_model, "googlenet_knn_model.pkl")
joblib.dump(scaler, "googlenet_knn_scaler.pkl")

print("_/_/ Model saved")

_/_/ Model saved


# Single Image Prediction

In [91]:
from tensorflow.keras.preprocessing import image

knn_model = joblib.load("googlenet_knn_model.pkl")
scaler = joblib.load("googlenet_knn_scaler.pkl")

class_indices = train_generator.class_indices
index_to_class = {v: k for k, v in class_indices.items()}

img_path = "dataset/test/A2-Sitting-down/235.png"
img = image.load_img(img_path, target_size=(224,224))

img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

features = feature_extractor.predict(img_array)
features = scaler.transform(features)

pred = knn_model.predict(features)[0]
confidence = np.max(knn_model.predict_proba(features))

print("Predicted Class:", index_to_class[pred])
print("Confidence:", confidence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
Predicted Class: A2-Sitting-down
Confidence: 0.8260869565217391
